# 51. What does an ordinary network do on this same stream?

## Why this exists

Notebook 50 found two things, and both of them have the same hole in them.

The first finding was about the shared feature direction. About 98 percent of
the study's feature matrix is the one row every picture has in common. An
ordinary network that was never trained reads the same way. An ordinary network
trained on ImageNet does not: it reads about 0.32.

The second finding was about direction of travel. Training an ordinary network
raises effective rank by more than ten times, from 16.8 to 224.6. Training this
hypernetwork lowers it by more than half, from 5.6 to 2.1. The two move opposite
ways.

The hole is the same in both. The trained network in notebook 50 was trained on
ImageNet: 1.28 million pictures and a thousand classes. The study's models saw
nine groups of twenty classes, one group at a time. So "trained" is not matched.
Any difference between the two sides could come from how much data was seen, or
from learning everything at once instead of one task after another, rather than
from where the weights come from.

This notebook closes that hole. It trains an ordinary network on the study's
own stream, with the study's own settings, and reads it at every boundary with
the study's own arithmetic.

## What changes, and what does not

| Held fixed | Changed |
| --- | --- |
| The nine groups, in order A, from the same ten-group split | The weights are ordinary parameters, not written by a generator from a 32 number code |
| Five epochs per group, batch 64, Adam at 1e-3 | There is no preservation term holding old tasks still |
| The same target architecture, 3 by 3 stride 1 stem, spatial mean | |
| The same three fixed 256 picture sets, same seed | |
| The same reading site, the 2048 inputs to the classifier | |
| The same effective rank, srank and mean share arithmetic | |

The network keeps one shared trunk and one small 20 way classifier per group.
That is the closest ordinary mirror of what the hypernetwork does: the
generator is shared across tasks and each task has its own code, so here the
trunk is shared across tasks and each task has its own head. All readings are
taken on the trunk, so the heads never enter a number.

## The extra question this run can answer for free

The study reads its indicators through a fixed set of BatchNorm statistics, the
clean ones a new task starts from, so that only the weights move between
boundaries. For an ordinary network that is an odd thing to do, because an
ordinary network has its own running statistics that it updates while training.

That difference could be the whole story. If a model is read through statistics
that no longer match its weights, its features may collapse for a reason that
has nothing to do with learning.

So every boundary here is read twice. Once with the network's own statistics,
which is what an ordinary practitioner would do. Once with the statistics reset
to their starting values, which is what the study's protocol does. If the two
disagree, the study's rank fall is partly about the reading route.

## What each answer would mean

**Effective rank rises over the stream.** Then the fall in the study is not a
consequence of training on this data in this order. It is about the
hypernetwork, and the opposite direction finding is a real result.

**Effective rank falls over the stream, in both buffer modes.** Then the fall is
about sequential training on small data, not about the generator. The paper
loses the opposite direction claim and gains a plainer one.

**The two buffer modes disagree.** Then the study's protocol is part of what it
measures, and that has to be said before any rank number is reported.

**Mean share stays near 0.98 after training.** Then the untrained signature is
about this data and this budget, not about the generator.

Nothing here is guessed in advance. Every outcome above is a result worth
writing down.

## 1. Setup

This one does need a GPU. It trains a ResNet50 for 7,065 steps per seed. On an
A100 that is roughly half an hour per seed; a smaller card will take longer.

Results are written per seed, so stopping after one seed leaves a usable file
and restarting skips what is already done.

In [ ]:
from pathlib import Path
import sys, os, json, subprocess, importlib.util

IN_COLAB = "google.colab" in sys.modules

# Importing uncle twice in one session gives two copies of the module with two
# copies of its constants, which is a confusing way to fail much later.
if any(name == "uncle" or name.startswith("uncle.") for name in sys.modules):
    raise RuntimeError("Restart the runtime before loading the code.")

if IN_COLAB:
    REPO = Path("/content/path-indicators")
    if not REPO.exists():
        subprocess.run(["git", "clone",
                        "https://github.com/sumitasthana/PATH.git", str(REPO)],
                       check=True)
    subprocess.run(["git", "-C", str(REPO), "fetch", "origin"], check=True)
    subprocess.run(["git", "-C", str(REPO), "checkout", "main"], check=True)
    subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only"], check=True)
else:
    REPO = next((p for p in [Path.cwd(), *Path.cwd().parents]
                 if (p / "uncle/task_partition_10x20.json").exists()), None)
    if REPO is None:
        raise FileNotFoundError("Open this from a checkout that has the ten-group split.")

for package in ("boto3", "tqdm"):
    if importlib.util.find_spec(package) is None and IN_COLAB:
        subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", package],
                       check=True)
sys.path.insert(0, str(REPO))

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader

from uncle.config import Config, PROBE_TASK, study_defaults
from uncle.hypernet import build_target
from uncle.plasticity_indicators import (SRANK_DELTA, THRESHOLDS,
                                         activation_sites,
                                         dormant_fraction,
                                         fixed_image_sets)
from uncle.storage import R2Store
from uncle.streams import build_tasks
from uncle.tasks import PARTITIONS

HEAD = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"],
                               text=True).strip()
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Running at:", HEAD)
print("Device:", DEVICE)
if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: no GPU found. This notebook trains; on CPU it will not finish.")

## 2. Settings

The training settings are not retyped here. They are read out of the same
`Config` the study builds, so they cannot drift away from it. The one thing
this notebook chooses is how many seeds to run.

In [ ]:
STUDY_ID = "plasticity_revised_v1"
BUCKET = "uncle-experiments"
ORDER = "order_A"

# Where this notebook's readings go. A separate key from notebooks 48 and 50,
# because these are different models again.
SAVE_KEY = f"uncle/indicator_protocol/{STUDY_ID}/plain_network_stream.jsonl"

# Notebook 48's hypernetwork readings, for the mean share comparison.
HYPERNET_KEY = f"uncle/indicator_protocol/{STUDY_ID}/code_sweep_and_centering.jsonl"

# The study's own live indicator logs, for the effective rank comparison.
# One file per stream, written while that stream trained.
def stream_key(seed):
    return f"uncle/indicators/{STUDY_ID}/{ORDER}/seed_{seed}.jsonl"

# Three seeds. One seed cannot show whether a direction of travel is
# consistent, and the study has fifteen streams on the other side. Three is the
# smallest number that can show agreement. Cut this to (0,) for a first look.
SEEDS = (0, 1, 2)

# The same 256 pictures per set that every boundary reading in the study used.
IMAGES_PER_SET = 256
READ_EUROSAT = True

# The Tiny ImageNet loader downloads only into a folder named after the
# dataset, so this path has to end in tiny-imagenet-200. EuroSAT makes its own
# subfolder, so it gets the parent instead. Same paths as notebooks 47 to 50.
DATA_ROOT = (Path("/content/data/tiny-imagenet-200") if IN_COLAB
             else REPO / "data/tiny-imagenet-200")
EUROSAT_ROOT = Path("/content/data") if IN_COLAB else REPO / "data"
CACHE = Path("./cache")
CACHE.mkdir(exist_ok=True)

# Build the study's config once, at seed 0, only to read settings off it. The
# per seed configs are built again inside the training loop.
REFERENCE_CONFIG = Config(**study_defaults(ORDER), seed=0, device=DEVICE)

# The learn requests, in the study's own order. Taking the order from the
# config rather than writing range(9) means a change to order A cannot leave
# this notebook quietly training a different stream.
STREAM_TASKS = [task for action, task in REFERENCE_CONFIG.requests
                if action == "learn"]
EPOCHS = REFERENCE_CONFIG.epochs
BATCH_SIZE = REFERENCE_CONFIG.batch_size
EVAL_BATCH_SIZE = REFERENCE_CONFIG.eval_batch_size
LEARNING_RATE = REFERENCE_CONFIG.learning_rate
CLASSES_PER_TASK = REFERENCE_CONFIG.classes_per_task

print("Order:", ORDER, "->", " ".join(STREAM_TASKS))
print(f"{EPOCHS} epochs per group, batch {BATCH_SIZE}, Adam at {LEARNING_RATE}")
print(f"{CLASSES_PER_TASK} classes per group, seeds {SEEDS}")


def secret(name):
    """Read a credential from Colab secrets, or the environment elsewhere."""
    if IN_COLAB:
        from google.colab import userdata
        return userdata.get(name)
    value = os.environ.get(name)
    if value is None:
        env = REPO / ".env"
        if env.exists():
            for line in env.read_text(encoding="utf-8").splitlines():
                if line.strip().startswith(name + "="):
                    return line.partition("=")[2].strip().strip('"').strip("'")
    return value


# R2Store holds a client, not a bucket name. `connect` builds the client from
# the four credentials and returns the store.
store = R2Store.connect(secret("R2_ENDPOINT"), BUCKET,
                        secret("R2_ACCESS_KEY_ID"), secret("R2_SECRET_ACCESS_KEY"))
print("Saving readings to:", SAVE_KEY)

## 3. The pictures

Two different sets of pictures are needed here, and keeping them apart matters.

**Training pictures.** The train split of each of the nine groups, used to
train the network. About 10,000 pictures per group.

**Reading pictures.** The same three fixed 256 picture sets every boundary
reading in the study used, drawn by the same function with the same seed. These
are never trained on. They are only pushed through the network to read it.

In [ ]:
from torch.utils.data import ConcatDataset

needed = tuple(STREAM_TASKS + [PROBE_TASK])
tasks = build_tasks(root=DATA_ROOT, partition_path=PARTITIONS[CLASSES_PER_TASK],
                    partition_seed=42, include=needed, download=True)
print("Loaded Tiny ImageNet pictures for", len(needed), "groups")
print("Training pictures in group", STREAM_TASKS[0] + ":",
      len(tasks[STREAM_TASKS[0]]["train"]))

sources = {
    "in_stream": ConcatDataset([tasks[name]["test"] for name in STREAM_TASKS]),
    "probe": tasks[PROBE_TASK]["test"],
}
if READ_EUROSAT:
    from torchvision.datasets import EuroSAT
    from torchvision.transforms import ToTensor
    sources["eurosat"] = EuroSAT(root=str(EUROSAT_ROOT), transform=ToTensor(),
                                 download=True)

image_sets = fixed_image_sets(sources, count=IMAGES_PER_SET)
for name, images in image_sets.items():
    print(f"{name:<10} {tuple(images.shape)}")

# A wrong picture size here would silently change every number below.
for name, images in image_sets.items():
    assert images.shape[1:] == (3, 64, 64), f"{name} is not 3 by 64 by 64"
    assert images.shape[0] == IMAGES_PER_SET, f"{name} has the wrong count"
print("Picture sets check out.")

## 4. The three measurements

Copied from notebook 50, which copied them from notebook 48, which copied them
from the library. The reason is the same every time: the centred and uncentred
numbers must go through identical arithmetic, or a difference between them
could come from the code rather than from the features.

**Effective rank** counts how many directions the features really use. It is
the exponential of the entropy of the normalised singular values, from Roy and
Vetterli 2007.

**srank** counts how many singular values are needed to reach 99 percent of
their total, from Kumar et al. 2021.

**Mean share** is how much of the whole matrix is the one average row, the part
every picture has in common, on a scale from 0 to 1.

In [ ]:
def rank_pair(matrix, centre):
    """Effective rank and srank of one feature matrix.

    `matrix` has one row per picture and 2048 columns. Setting `centre` to True
    subtracts the average row first, which removes what every picture has in
    common and leaves only how the pictures differ.
    """
    work = matrix.double()
    if centre:
        work = work - work.mean(dim=0, keepdim=True)
    values = torch.linalg.svdvals(work)
    total = values.sum()
    if total <= 0:
        return float("nan"), 0
    weights = values / total
    weights = weights[weights > 0]
    effective = float(torch.exp(-(weights * weights.log()).sum()).item())
    shares = torch.cumsum(values, 0) / total
    return effective, int((shares < 1 - SRANK_DELTA).sum().item()) + 1


def mean_share(matrix):
    """How much of the matrix is the one row every picture shares.

    The average row times the square root of the number of rows is the size of
    the rank one matrix made by repeating that average. Dividing by the size of
    the whole matrix gives the share, between 0 and 1.
    """
    work = matrix.double()
    norm = float(work.norm())
    if norm <= 0:
        return float("nan")
    return float(work.mean(dim=0).norm() * (work.shape[0] ** 0.5) / norm)

## 5. The network, and the two ways of reading it

The trunk is the study's own target network with its classifier removed, so it
returns the 2048 numbers the study reads. Each group gets its own 20 way
classifier on top. Only the trunk is ever read.

`build_target` turns gradients off, because in the study this network is never
trained: the generator writes fresh weights on every forward pass. Here it is
the thing being trained, so gradients go back on.

The two buffer modes are the extra question from the top of this notebook. A
BatchNorm layer carries running statistics, which are averages of what it has
seen. The study reads through a fixed clean set of them. An ordinary network
reads through its own. Both are taken at every boundary.

In [ ]:
def build_plain(seed):
    """One shared trunk and one 20 way classifier per group.

    The seed is set before the trunk is built, so two runs at the same seed
    start from the same weights.
    """
    torch.manual_seed(seed)
    config = Config(**study_defaults(ORDER), seed=seed, device=DEVICE)
    trunk = build_target(config)
    trunk.fc = nn.Identity()
    # build_target leaves gradients off because the hypernetwork never trains
    # this network directly. Here it is the thing being trained.
    trunk.requires_grad_(True)
    heads = nn.ModuleDict({name: nn.Linear(2048, CLASSES_PER_TASK)
                           for name in STREAM_TASKS})
    return trunk.to(DEVICE), heads.to(DEVICE)


def buffer_snapshot(model):
    """A copy of every BatchNorm running statistic, to put back later."""
    return {name: value.detach().clone() for name, value in model.named_buffers()}


def load_buffers(model, state):
    """Write a saved set of running statistics back into the model."""
    with torch.no_grad():
        for name, value in model.named_buffers():
            value.copy_(state[name])


@torch.no_grad()
def read_features(model, images, batch_size=64):
    """The 2048 classifier inputs for one set of pictures.

    Evaluation mode is forced. In training mode a BatchNorm layer rescales
    using whatever batch is in front of it, which can revive a nearly dead
    signal and hide the thing being measured. The study forces it for the same
    reason.
    """
    was_training = model.training
    model.eval()
    try:
        collected = []
        for start in range(0, len(images), batch_size):
            batch = images[start:start + batch_size]
            collected.append(model(batch.to(DEVICE)).float().cpu())
        return torch.cat(collected)
    finally:
        model.train(was_training)


@torch.no_grad()
def read_dormant(model, images, batch_size=64):
    """The share of units that have gone quiet, at three cut-offs.

    A unit is called dormant when its mean absolute activation is small beside
    the mean over all units at the same place in the network. Sokar et al. 2023
    define it that way and report three cut-offs, which is what THRESHOLDS
    holds. The counting is the library's own, so these numbers go through the
    same arithmetic as the study's published ones.

    This reads every ReLU in the network, not only the 2048 numbers at the
    classifier, so it is the broadest of the three indicators. The model is
    left in the mode it was found in.
    """
    was_training = model.training
    model.eval()
    try:
        def run(reset):
            for start in range(0, len(images), batch_size):
                # The sums are keyed by the position of each ReLU call, so the
                # counter goes back to zero at the start of every batch.
                reset()
                model(images[start:start + batch_size].to(DEVICE))
        return dormant_fraction(activation_sites(model, run))
    finally:
        model.train(was_training)


def read_boundary(trunk, start_buffers, seed, t, task):
    """Every indicator at one boundary, both buffer modes, all picture sets.

    `own` statistics are the ones the network has built up while training, which
    is what an ordinary user would read through. `start` statistics are the
    untouched ones it was born with, which is the route the study uses.
    """
    rows = []
    live = buffer_snapshot(trunk)
    for mode in ("own", "start"):
        load_buffers(trunk, live if mode == "own" else start_buffers)
        for set_name, images in image_sets.items():
            matrix = read_features(trunk, images)
            # 2048 is the width of a ResNet50's last layer. A different width
            # here means the features are being read in the wrong place.
            assert matrix.shape == (IMAGES_PER_SET, 2048), matrix.shape
            raw_e, raw_s = rank_pair(matrix, centre=False)
            mid_e, mid_s = rank_pair(matrix, centre=True)
            rows.append(dict(
                study=STUDY_ID, order=ORDER, seed=seed, t=t, task=task,
                buffers=mode, images=set_name,
                effective_rank=raw_e, srank=raw_s,
                effective_rank_centred=mid_e, srank_centred=mid_s,
                mean_share=mean_share(matrix),
                **read_dormant(trunk, images),
                n_images=int(matrix.shape[0]),
                code_revision=HEAD, device=DEVICE))
    load_buffers(trunk, live)
    return rows

## 6. Training one group

Five epochs, batch 64, Adam at 1e-3, a fresh optimizer per group. These are the
study's values, read off its config rather than typed here.

There is no term holding the earlier groups still. The study has one, weighted
by beta. Leaving it out is on purpose: this is a plain network trained the
ordinary way, and adding a preservation term would be a different experiment.
It does mean the two sides differ in one way beyond where the weights come
from, and that has to be said when the result is written up.

In [ ]:
from uncle.telemetry import progress_bar


def train_group(trunk, head, dataset, seed, task):
    """Train the shared trunk and this group's classifier for five epochs.

    Returns the mean loss of each epoch, which is a cheap way to see that
    learning happened at all.
    """
    generator = torch.Generator().manual_seed(seed)
    loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True,
                        generator=generator, num_workers=2, drop_last=False)
    optimizer = torch.optim.Adam([*trunk.parameters(), *head.parameters()],
                                 lr=LEARNING_RATE)
    trunk.train()
    head.train()
    bar = progress_bar(EPOCHS * len(loader), f"Learn {task}", True, leave=False)
    losses = []
    try:
        for epoch in range(EPOCHS):
            running = 0.0
            for images, labels in loader:
                scores = head(trunk(images.to(DEVICE)))
                loss = F.cross_entropy(scores, labels.to(DEVICE))
                if not torch.isfinite(loss):
                    raise FloatingPointError("Non-finite loss.")
                optimizer.zero_grad(set_to_none=True)
                loss.backward()
                optimizer.step()
                running += loss.item()
                bar.update(1)
                bar.set_postfix_str(f"epoch {epoch+1}/{EPOCHS}, loss {loss.item():.3f}")
            losses.append(running / len(loader))
    finally:
        bar.close()
    return losses


@torch.no_grad()
def accuracy(trunk, head, dataset):
    """Percent correct on one group's test split, through that group's head."""
    trunk.eval()
    head.eval()
    correct = total = 0
    for images, labels in DataLoader(dataset, batch_size=EVAL_BATCH_SIZE):
        scores = head(trunk(images.to(DEVICE)))
        correct += (scores.argmax(1) == labels.to(DEVICE)).sum().item()
        total += labels.numel()
    return 100.0 * correct / total

## 7. Run the stream

One seed at a time. Boundary 0 is the network before any training, which is
what notebook 50 already read as "study target, random weights". Boundary 1 is
after the first group, and so on to boundary 9.

Boundary 1 is worth noticing. The study cannot re-read its own boundary 1,
because only boundaries 4, 9 and 10 were saved as checkpoints. Here the reading
happens while the run is going, so boundary 1 comes for free. That is where the
study's whole rank fall happens, so having an ordinary network's value at the
same place is the sharpest single comparison in this notebook.

A seed already in the saved file is skipped, so a stopped run can be restarted.

In [ ]:
import time

# Pick up anything already done, so a stopped run does not repeat a seed.
rows = []
if store.info(SAVE_KEY) is not None:
    text = store.download(SAVE_KEY, CACHE).read_text(encoding="utf-8")
    rows = [json.loads(line) for line in text.splitlines() if line.strip()]
    print(f"Found {len(rows)} saved readings from earlier runs.")
done_seeds = {r["seed"] for r in rows if r.get("t") == len(STREAM_TASKS)}
if done_seeds:
    print("Seeds already finished:", sorted(done_seeds))

accuracies = []
LOCAL = CACHE / "plain_network_stream_v2.jsonl"

for seed in SEEDS:
    if seed in done_seeds:
        print(f"seed {seed}: already done, skipping")
        continue
    started = time.perf_counter()
    print(f"\n=== seed {seed} ===")
    trunk, heads = build_plain(seed)
    # The statistics the network was born with. The study reads every boundary
    # through its own equivalent of these.
    start_buffers = buffer_snapshot(trunk)

    rows += read_boundary(trunk, start_buffers, seed, 0, None)
    print(f"  t=0  read")

    for position, task in enumerate(STREAM_TASKS, start=1):
        losses = train_group(trunk, heads[task], tasks[task]["train"], seed, task)
        just_learned = accuracy(trunk, heads[task], tasks[task]["test"])
        rows += read_boundary(trunk, start_buffers, seed, position, task)
        accuracies.append(dict(seed=seed, t=position, task=task,
                               just_learned=just_learned,
                               first_epoch_loss=losses[0],
                               last_epoch_loss=losses[-1]))
        print(f"  t={position}  group {task}: loss {losses[0]:.3f} -> "
              f"{losses[-1]:.3f}, test {just_learned:.1f}%")

    # How much was forgotten. Every group scored again at the end of the stream,
    # each through its own head. The trunk has moved on since each was trained.
    final = {task: accuracy(trunk, heads[task], tasks[task]["test"])
             for task in STREAM_TASKS}
    for row in accuracies:
        if row["seed"] == seed:
            row["at_end"] = final[row["task"]]
    print("  end of stream, each group through its own head:")
    print("   ", "  ".join(f"{t}:{v:.1f}%" for t, v in final.items()))

    # Save after every seed, so a run stopped halfway still leaves usable work.
    LOCAL.write_text("".join(json.dumps(r) + "\n" for r in rows), encoding="utf-8")
    store.upload(LOCAL, SAVE_KEY, overwrite=True)
    print(f"  saved, {(time.perf_counter() - started) / 60:.1f} minutes")

    del trunk, heads
    if DEVICE == "cuda":
        torch.cuda.empty_cache()

print(f"\n{len(rows)} readings in the file.")

## 8. Did it actually learn?

Before any rank number means anything, the network has to have learned the
groups. If it did not, the comparison is void: an ordinary network that failed
to train is not a control for anything.

Chance is 5 percent. The hypernetwork's own streams reach the high thirties on
a group they have just learned.

In [ ]:
if accuracies:
    print(f"{'seed':>5} {'t':>3} {'group':>6} {'loss first':>11} {'loss last':>10} "
          f"{'just learned':>13} {'at end':>8}")
    for row in accuracies:
        print(f"{row['seed']:>5} {row['t']:>3} {row['task']:>6} "
              f"{row['first_epoch_loss']:>11.3f} {row['last_epoch_loss']:>10.3f} "
              f"{row['just_learned']:>12.1f}% {row.get('at_end', float('nan')):>7.1f}%")
    learned = [r["just_learned"] for r in accuracies]
    kept = [r["at_end"] for r in accuracies if "at_end" in r]
    print(f"\nmean when just learned {sum(learned)/len(learned):.1f}%, "
          f"mean at the end of the stream {sum(kept)/len(kept):.1f}%, chance 5.0%")
    if sum(learned) / len(learned) < 10:
        print("WARNING: this network did not learn the groups. Nothing below is a "
              "valid control.")
else:
    print("No training happened in this session. Accuracies live in the run above.")

## 9. What the indicators did over the stream

The first table is the whole answer. It is effective rank on the in-stream
pictures at every boundary, averaged over the seeds that ran, in both buffer
modes.

Read the first column against the study's own fall from 5.563 at boundary 0 to
1.723 at boundary 1 and 2.094 at boundary 9.

In [ ]:
from collections import defaultdict


def mean(values):
    values = [v for v in values if v == v]
    return sum(values) / len(values) if values else float("nan")


def grid(rows, field, images="in_stream"):
    """One row per boundary, one column per buffer mode."""
    out = defaultdict(dict)
    for mode in ("own", "start"):
        for t in range(len(STREAM_TASKS) + 1):
            values = [r[field] for r in rows
                      if r["buffers"] == mode and r["images"] == images
                      and r["t"] == t]
            out[t][mode] = mean(values)
    return out


def show(title, field, fmt="8.3f"):
    table = grid(rows, field)
    print()
    print(title, "on in-stream pictures")
    print(f"{'boundary':>9} {'own stats':>12} {'start stats':>12} {'seeds':>7}")
    for t in sorted(table):
        n = len({r["seed"] for r in rows if r["t"] == t})
        print(f"{t:>9} {table[t]['own']:>12.4f} {table[t]['start']:>12.4f} {n:>7}")
    return table


eff = show("EFFECTIVE RANK", "effective_rank")
share = show("MEAN SHARE", "mean_share")
centred = show("EFFECTIVE RANK, shared row removed", "effective_rank_centred")

## 10. The verdict

Three questions, answered one at a time from the tables above and from the
study's own saved files.

In [ ]:
LAST = len(STREAM_TASKS)

# ---- question 1: which way does an ordinary network move? ----------------
print("1. DIRECTION OF TRAVEL, effective rank on in-stream pictures")
print()
for mode in ("own", "start"):
    a, b, c = eff[0][mode], eff[1][mode], eff[LAST][mode]
    print(f"   {mode + ' statistics':<18} t=0 {a:8.3f}   t=1 {b:8.3f}   "
          f"t={LAST} {c:8.3f}   change {c - a:+8.3f}")

# The study's own numbers, read from the live indicator logs rather than typed.
hyper_t = defaultdict(list)
for seed in range(5):
    key = stream_key(seed)
    if store.info(key) is None:
        continue
    text = store.download(key, CACHE).read_text(encoding="utf-8")
    for line in text.splitlines():
        if line.strip():
            row = json.loads(line)
            if "effective_rank_in_stream" in row:
                hyper_t[row["t"]].append(row["effective_rank_in_stream"])
if hyper_t:
    a, b, c = mean(hyper_t[0]), mean(hyper_t[1]), mean(hyper_t[LAST])
    print(f"   {'hypernetwork':<18} t=0 {a:8.3f}   t=1 {b:8.3f}   "
          f"t={LAST} {c:8.3f}   change {c - a:+8.3f}   "
          f"({len(hyper_t[0])} streams)")
    # The study reads every model through `trainer.buffer_template`, which holds
    # the statistics a freshly built ResNet50 has. That is the `start` route
    # here, so `start` is the only column that can be set beside the study's
    # own numbers. The `own` column is a separate question and gets its own
    # sentence below, never the same one.
    matched_change = eff[LAST]["start"] - eff[0]["start"]
    own_change = eff[LAST]["own"] - eff[0]["own"]
    hyper_change = c - a
    print()
    if matched_change < 0 and hyper_change < 0 and own_change > 0:
        print("   THE ROUTE, NOT THE STREAM. On the matched route an ordinary")
        print("   network falls here too, so the opposite direction claim has to")
        print("   come out of the paper. But the same network, at the same")
        print("   boundaries, read through its own statistics rises by")
        print(f"   {own_change:+.1f}. Sequential training on these nine groups does not")
        print("   reduce effective rank; it raises it by a large factor. The fall")
        print("   is what happens when weights that have travelled are read")
        print("   through BatchNorm statistics that never moved.")
    elif matched_change > 0 and hyper_change < 0:
        print("   OPPOSITE. On the matched route an ordinary network moves the")
        print("   other way, so the fall is about the hypernetwork and the")
        print("   ImageNet confound is closed.")
    elif matched_change < 0 and hyper_change < 0:
        print("   SAME on the matched route, and the own-statistics route does")
        print("   not separate the two either. Report both columns and say so.")
    else:
        print("   Neither clean case. Report both numbers and say what is unclear.")
else:
    print("   The study's indicator logs were not found, so no comparison is made.")

# ---- question 2: does the reading route matter? --------------------------
print()
print("2. DOES THE READING ROUTE MATTER?")
print()
gap = abs(eff[LAST]["own"] - eff[LAST]["start"])
relative = gap / max(eff[LAST]["own"], eff[LAST]["start"])
print(f"   At t={LAST}, own statistics read {eff[LAST]['own']:.3f} and start")
print(f"   statistics read {eff[LAST]['start']:.3f}. That is a gap of {gap:.3f}, "
      f"or {100 * relative:.0f} percent.")
if relative > 0.2:
    print("   The two routes disagree. The study's fixed-buffer protocol is part")
    print("   of what it measures, and the paper must say so before any rank")
    print("   number is reported.")
else:
    print("   The two routes agree. Reading through fixed statistics is not what")
    print("   produces the study's numbers.")

# ---- question 3: does the untrained signature survive training? ----------
print()
print("3. DOES THE 0.98 SIGNATURE SURVIVE TRAINING ON THIS STREAM?")
print()
# The study's 0.9811 is a `start` route number, so only the `start` column can
# be set beside it. Putting it next to the `own` column would be two different
# reading routes inside one comparison.
print(f"   start statistics  t=0 {share[0]['start']:.4f}, "
      f"t={LAST} {share[LAST]['start']:.4f}   (the study's route)")
print(f"   own statistics    t=0 {share[0]['own']:.4f}, "
      f"t={LAST} {share[LAST]['own']:.4f}")
print("   The study's own models read about 0.9811 after nine groups, through")
print("   the same fixed statistics as the start line above.")
if share[LAST]["start"] > 0.9:
    print("   On the matched route the signature survives. An ordinary network")
    print("   keeps it too, so holding a high shared row is not a property of")
    print("   the generator. The own statistics line shows what the route hides:")
    print("   statistics measured from the model's own weights bring it down.")
else:
    print("   On the matched route the signature does not survive. An ordinary")
    print("   network loses it where the study's models keep it, which is a real")
    print("   difference between the two with the data now matched.")

## 11. What this still does not settle

**No preservation term.** The study's streams carry a term, weighted by beta,
that holds the earlier groups still while a new one is learned. This network has
none. If the two sides disagree, that term is a second possible cause alongside
where the weights come from, and the honest write-up names both.

**One architecture, one order, three seeds.** Order A only. A different ordering
would not be expected to change a direction of travel, but it has not been
checked here.

**The heads are not the codes.** A 20 way linear classifier per group is the
closest ordinary mirror of a per-task code, but it is not the same object. It
has 40,980 numbers; a task code has 32. Nothing is read through the heads, so
they do not enter a number, but they do take some of the learning load that the
hypernetwork has to put into the generator.

**srank is a whole number.** At 256 pictures it cannot exceed 256, so a centred
srank near 256 means the measure has run out of room rather than that the
features use exactly that many directions.